# `pheno_00_table_summary` — phenotype table summary, with Figure 3A/3B

Run on the Terra Workbench VM. Reads one phenotype CSV, the covariate table,
`phecodeX_info.csv` and chr1 FLARE global ancestry; writes tables and figures
under `pheno_summary/`.

Panels A and B are drawn by `manuscript_figures`, not re-implemented:
`scripts/make_fig3_case_counts.py` writes the counts, `fig3_sim.build_coverage`
reads them, `panel_matrix_h` and `fig3_interim.panel_cutoff` draw them, and the
floor is `fig3_sim.MIN_CASES`. `assert not cov.simulated` keeps the package's
layout placeholders out.

Counts file names follow the cohort, so a `legacy` run cannot overwrite the
current figure's inputs:

| Cohort | Files |
|---|---|
| `selector` (`analysis_samples.txt` + `pheno_cov.tsv`, the cohort the scan is fitted on) | `fig3_phecode_case_counts_current.tsv`, `fig3_cohort_summary_current.json` |
| `legacy` (releasable PacBio × phenotype table × defined `sex_at_birth` × complete covariates) | `fig3_phecode_case_counts.tsv`, `fig3_cohort_summary.json` |

Input paths are literal, from `manifest.txt.gz`; each takes an environment
override: `PHENO_PHENOTYPE_GCS`, `PHENO_COVARIATES_GCS`,
`PHENO_PHECODE_INFO_GCS`, `PHENO_GLOBAL_ANC_GCS`, `PHENO_SELECTION_PREFIX`.
Also `PHENO_SELECTION_DIR`, `PHENO_COHORT` (`auto|selector|legacy`),
`PHENO_FIGURE_CODE_DIR`, `PHENO_FIGURE_CODE_GCS`, `PHENO_FIGURE_ROOT`,
`PHENO_SUMMARY_DIR`,
`PHENO_BUCKET_PREFIX`, `PHENO_MAX_R2`.

## Bootstrap `scripts/`

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError("scripts/ not found locally and WORKSPACE_BUCKET is unset.")
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"])
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook
SCRIPTS = init_notebook("select_phenotypes.py", "make_fig3_case_counts.py")


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command and fail on a non-zero exit (get_ipython().system() does not)."""
    print(cmd)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


import select_phenotypes as sp
import make_fig3_case_counts as mf

print("scripts  :", SCRIPTS)
print("selector :", (SCRIPTS / "select_phenotypes.py").resolve())
print("counts   :", (SCRIPTS / "make_fig3_case_counts.py").resolve())

## Config

In [ ]:
import json
import gzip
import shutil
import types

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paths on Terra, from manifest.txt.gz; each takes an environment override.
WS = os.environ.get("WORKSPACE_BUCKET", "gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09").rstrip("/")

PHENOTYPE_URI = os.environ.get("PHENO_PHENOTYPE_GCS", f"{WS}/felix/phenotypes/AoU_Phase2_Phenotype.csv.gz")
COVARIATES_URI = os.environ.get("PHENO_COVARIATES_GCS", f"{WS}/covariates/covariates.source_rebuilt.csv.gz")
PHECODE_URI = os.environ.get("PHENO_PHECODE_INFO_GCS", f"{WS}/refs/phecodeX_info.csv")
# chr1 FLARE global ancestry (SAMPLE eas amr eur afr sas), behind the hap_* columns:
# the global.anc.gz MergePopulationFlare writes beside the chr1 anc VCF, so it is pinned to
# the same run (FlareByPopulation submission 6db7352b, chr1 workflow 24078b2c).
FLARE_CHR1_CALL = (f"{WS}/submissions/6db7352b-1190-4cd4-8fc4-4e768786f4e7/FlareByPopulation"
                   "/24078b2c-d1ba-400b-a7c0-864d21665a25/call-MergePopulationFlare")
GLOBAL_ANC_URI = os.environ.get("PHENO_GLOBAL_ANC_GCS",
                                f"{FLARE_CHR1_CALL}/lai_prod.aou_lr_phase2_v1.chr1.global.anc.gz")

WORK = Path(os.environ.get("PHENO_SUMMARY_DIR", "pheno_summary")).resolve()
TABLES, FIGURES = WORK / "tables", WORK / "figures"
for _d in (WORK, TABLES, FIGURES):
    _d.mkdir(parents=True, exist_ok=True)

MIN_PREVALENCE_CASES = 20  # blank a chapter-prevalence cell below this, as fig_phenotypes.py does

SELECTION_PREFIX = os.environ.get("PHENO_SELECTION_PREFIX", f"{WS}/felix/phenotypes/wgs").rstrip("/")
COHORT_REQUEST = os.environ.get("PHENO_COHORT", "auto").strip().lower()
assert COHORT_REQUEST in ("auto", "selector", "legacy"), COHORT_REQUEST
BUCKET_PREFIX = os.environ.get("PHENO_BUCKET_PREFIX", f"{WS}/pheno_summary").rstrip("/")

for label, uri in [("phenotype", PHENOTYPE_URI), ("covariates", COVARIATES_URI), ("phecodeX", PHECODE_URI),
                   ("global anc", GLOBAL_ANC_URI), ("selection", SELECTION_PREFIX)]:
    print(f"{label:11s}: {uri}")
print(f"{'cohort':11s}: {COHORT_REQUEST}")
print(f"{'work dir':11s}: {WORK}")
print(f"{'upload to':11s}: {BUCKET_PREFIX}")

## Figure code

All six modules in the import closure must sit in one directory: a local
checkout, a copy rsynced to `$WORKSPACE_BUCKET/manuscript_figures/`, or a clone
of `main` — tried in that order, with rejected candidates printed. A clone is the
weakest: `fig3_interim.py` is untracked, and `fig3_scan.py`,
`fig3_scan_merged.py` and `fig3_sim.py` carry uncommitted edits.

`style.ROOT` is resolved at import, so `AOU_LR_MANUSCRIPT_ROOT` is set first and
cached modules are dropped (the cell is re-runnable). The package `__init__`
imports every figure module in the repo, so a stub package module is installed
and only the closure is executed.

In [ ]:
DEFAULT_REPO_URL = "https://github.com/kvg/aou-lr-phase-2.git"
# The modules imported below, including everything they import in turn.
MF_MODULES = ("style.py", "fig3_sim.py", "fig3_scan.py", "fig3_scan_chapter_cards.py",
              "fig3_scan_merged.py", "fig3_interim.py")
MF_PREFIX = os.environ.get("PHENO_FIGURE_CODE_GCS", f"{WS}/manuscript_figures").rstrip("/")
CODE_DIR = Path(os.environ.get("PHENO_CODE_DIR", "pheno_code")).resolve()  # fetched code, never uploaded


def mf_complete(d: Path) -> bool:
    return d.is_dir() and all((d / m).is_file() for m in MF_MODULES)


def find_figure_code() -> Path:
    """The manuscript_figures directory holding all of MF_MODULES.

    A local checkout first, then a copy rsynced to the bucket, then a clone of
    main; the fetches are only attempted if nothing local matched.
    """
    tried = []
    env = os.environ.get("PHENO_FIGURE_CODE_DIR", "").strip()
    for c in ([Path(env)] if env else []) + [c / "manuscript_figures" for c in
              (SCRIPTS.parent, Path.cwd(), Path.cwd().parent, Path.cwd() / "aou-lr-phase-2",
               Path.home() / "aou-lr-phase-2")]:
        if mf_complete(c):
            return c.resolve()
        tried.append(c)

    staged = CODE_DIR / "manuscript_figures"   # a copy rsynced to the bucket
    if not mf_complete(staged):
        sh(f"gsutil -m rsync -r {MF_PREFIX}/ {staged}/", check=False)
    if mf_complete(staged):
        return staged.resolve()
    tried.append(staged)

    clone = CODE_DIR / "aou-lr-phase-2" / "manuscript_figures"   # last resort: a clone of main
    if not mf_complete(clone):
        sh(f"git clone --depth 1 {os.environ.get('PHENO_REPO_URL', DEFAULT_REPO_URL)} {clone.parent}",
           check=False)
    if mf_complete(clone):
        return clone.resolve()
    tried.append(clone)

    for c in tried:
        missing = [m for m in MF_MODULES if not (c / m).is_file()]
        print(f"  {c}: " + ("absent" if len(missing) == len(MF_MODULES) else f"missing {missing}"))
    raise SystemExit(
        "No directory has all of " + ", ".join(MF_MODULES) + ".\n"
        "fig3_interim.py is untracked in the code repo, so a clone of main does not carry it, and\n"
        "fig3_scan.py / fig3_scan_merged.py / fig3_sim.py may have uncommitted edits. Either commit\n"
        "and push them and re-run 00_sync_repo, or stage the directory from a machine with bucket\n"
        f"access:\n  gsutil -m rsync -r manuscript_figures/ {MF_PREFIX}/\n"
        "PHENO_FIGURE_CODE_DIR overrides with a local directory.")


MF_DIR = find_figure_code()
FIG_ROOT = Path(os.environ.get("PHENO_FIGURE_ROOT", str(WORK / "figure_data_root"))).resolve()
FIG_DATA = FIG_ROOT / "data"
FIG_DATA.mkdir(parents=True, exist_ok=True)
os.environ["AOU_LR_MANUSCRIPT_ROOT"] = str(FIG_ROOT)

for _m in [m for m in sys.modules if m == "manuscript_figures" or m.startswith("manuscript_figures.")]:
    sys.modules.pop(_m)
_pkg = types.ModuleType("manuscript_figures")
_pkg.__path__ = [str(MF_DIR)]
sys.modules["manuscript_figures"] = _pkg

from manuscript_figures import fig3_interim as I
from manuscript_figures import fig3_scan_merged as M
from manuscript_figures import fig3_sim as S
from manuscript_figures.fig3_scan_chapter_cards import panel_matrix_h
from manuscript_figures.style import ANC_COLORS, ANC_TEXT, DATADIR, FIG_W, new_figure

assert DATADIR == FIG_DATA, (DATADIR, FIG_DATA)  # the env var took effect before the import

MIN_CASES, MIN_HAP = S.MIN_CASES, S.MIN_HAP
GROUPS, LOCAL = list(S.GROUP_ANC), list(S.LOCAL_ANC)

print("figure code:", MF_DIR)
print("figure data:", FIG_DATA)
print("min cases  :", MIN_CASES, f"(min case haplotypes {MIN_HAP})   [fig3_sim.MIN_CASES]")
print("groups     :", GROUPS)
print("local anc  :", LOCAL)

## Localize the inputs

In [ ]:
def localize(uri: str, dest: str | Path | None = None) -> Path:
    """Local path for a gs:// URI or an already-local path."""
    if not uri.startswith("gs://"):
        p = Path(uri).expanduser()
        assert p.is_file(), f"{p} does not exist"
        if dest is not None and Path(dest).resolve() != p.resolve():
            shutil.copyfile(p, dest)
            return Path(dest).resolve()
        return p.resolve()
    out = Path(dest or uri.rsplit("/", 1)[-1])
    if not out.exists():
        sh(f"gsutil cp {uri} {out}")
    assert out.is_file() and out.stat().st_size > 0, out
    return out.resolve()


PHENOTYPE_CSV = localize(PHENOTYPE_URI)
COVARIATES_CSV = localize(COVARIATES_URI)
PHECODE_CSV = localize(PHECODE_URI, FIG_DATA / "phecodeX_info.csv")
GLOBAL_ANC = localize(GLOBAL_ANC_URI)

# The selector cohort: analysis_samples.txt + the sex column of pheno_cov.tsv.
# make_fig3_case_counts.py reads both out of one directory.
SELECTION_DIR = Path(os.environ.get("PHENO_SELECTION_DIR", "pheno_selection")).resolve()
SELECTION_DIR.mkdir(parents=True, exist_ok=True)
for name in ("analysis_samples.txt", "pheno_cov.tsv", "selected_phenotypes.txt", "selected_phenotype_stats.tsv"):
    dest = SELECTION_DIR / name
    if not dest.exists() and SELECTION_PREFIX.startswith("gs://"):
        sh(f"gsutil cp {SELECTION_PREFIX}/{name} {dest}", check=False)

have_selection = (SELECTION_DIR / "analysis_samples.txt").is_file() and (SELECTION_DIR / "pheno_cov.tsv").is_file()
COHORT = COHORT_REQUEST if COHORT_REQUEST != "auto" else ("selector" if have_selection else "legacy")
if COHORT == "selector" and not have_selection:
    raise SystemExit(f"PHENO_COHORT=selector needs analysis_samples.txt and pheno_cov.tsv under {SELECTION_DIR} "
                     f"(looked in {SELECTION_PREFIX}). Set PHENO_SELECTION_PREFIX, or use PHENO_COHORT=legacy.")

for label, p in [("phenotype", PHENOTYPE_CSV), ("covariates", COVARIATES_CSV), ("phecodeX", PHECODE_CSV),
                 ("global anc", GLOBAL_ANC)]:
    print(f"{label:11s}: {p}  ({p.stat().st_size / 1e6:,.1f} MB)")
print(f"{'cohort':11s}: {COHORT}" + ("" if COHORT == "selector" else
      "  (covariate-derived; NOT the cohort the scan is fitted on)"))

## 1. Shape and phecode dialect

In [ ]:
pheno = pd.read_csv(PHENOTYPE_CSV, dtype=str, low_memory=False)
pheno = pheno.drop(columns=[c for c in pheno.columns if c.startswith("Unnamed") or c == ""], errors="ignore")
assert "person_id" in pheno.columns, f"no person_id column; got {list(pheno.columns)[:8]}"
pheno["person_id"] = pheno["person_id"].astype(str)
assert pheno["person_id"].is_unique, "duplicate person_id in the phenotype table"
pheno = pheno.set_index("person_id")

PHE_COLS = [c for c in pheno.columns if c != "sex"]  # the selector's candidate set: every column but id and sex
CODE_OF = {c: sp.phecode_of(c) for c in PHE_COLS}

PHECODE_ROWS = S.load_phecodes()
CHAPTER = {r["phecode"]: r["category"] for r in PHECODE_ROWS}
CODE_NAME = {r["phecode"]: r["phecode_string"] for r in PHECODE_ROWS}
CHAPTER_SIZE = pd.Series([r["category"] for r in PHECODE_ROWS]).value_counts()
PHECODE_SEX = sp.read_phecode_sex(PHECODE_CSV)

unparsed = [c for c, k in CODE_OF.items() if not k]
unknown = sorted({k for k in CODE_OF.values() if k and k not in CHAPTER})
dupes = pd.Series([k for k in CODE_OF.values() if k]).value_counts()
dupes = dupes[dupes > 1]
values = pd.Series(np.concatenate([pheno[c].to_numpy(dtype=object) for c in PHE_COLS[:50]]))
vocab = values.fillna("<missing>").value_counts()

shape = pd.Series({
    "participants": len(pheno),
    "phenotype columns": len(PHE_COLS),
    "columns parsed as a phecode": len(PHE_COLS) - len(unparsed),
    "codes in the PhecodeX table": len([k for k in CODE_OF.values() if k in CHAPTER]),
    "codes NOT in the PhecodeX table": len(unknown),
    "PhecodeX codes defined": len(CHAPTER),
    "PhecodeX codes absent from this table": len(set(CHAPTER) - {k for k in CODE_OF.values() if k}),
    "PhecodeX chapters covered": len({CHAPTER[k] for k in CODE_OF.values() if k in CHAPTER}),
    "sex column present": "sex" in pheno.columns,
}, dtype=object)
print(shape.to_string())
print("\nraw value vocabulary (first 50 phenotype columns):")
print(vocab.to_string())
if unparsed:
    print(f"\n{len(unparsed)} columns are not phecode names (ignored by the selector too): {unparsed[:10]}")
if unknown:
    print(f"\n{len(unknown)} codes are absent from phecodeX_info.csv and carry no chapter: {unknown[:10]}")
    print("make_fig3_case_counts.py asserts on these; they must be resolved before the counts step.")
if len(dupes):
    print(f"\n{len(dupes)} phecodes are carried by more than one column, so they are counted twice:")
    print(dupes.head(10).to_string())

## 2. Cohort attrition

In [ ]:
cov = mf.load_covariates(COVARIATES_CSV)
cov_df = pd.DataFrame.from_dict(cov, orient="index")
cov_df.index.name = "research_id"

in_pheno = cov_df.index.intersection(pheno.index)
complete = cov_df.loc[in_pheno][cov_df.loc[in_pheno, "complete"]].index
sex_defined = [i for i in complete if cov_df.at[i, "sex_at_birth"] in ("Female", "Male")]

steps = [
    ("rows in the phenotype table", len(pheno)),
    ("releasable PacBio in the covariate table", len(cov_df)),
    ("  ... and in the phenotype table", len(in_pheno)),
    ("  ... and complete age / coverage / PC1-10", len(complete)),
    ("  ... and with a defined sex_at_birth (legacy cohort)", len(sex_defined)),
]

if COHORT == "selector":
    IDS = [s.strip() for s in (SELECTION_DIR / "analysis_samples.txt").read_text().splitlines() if s.strip()]
    sel = pd.read_csv(SELECTION_DIR / "pheno_cov.tsv", sep="\t", usecols=["ID", "sex"], dtype={"ID": str})
    SEX = sel.set_index("ID")["sex"].reindex(IDS)
    assert SEX.notna().all(), "selector sex must be defined for every analysis sample"
    steps.append(("select_phenotypes.py analysis cohort (used here)", len(IDS)))
else:
    IDS = sorted(sex_defined)
    SEX = pd.Series({i: (0.0 if cov_df.at[i, "sex_at_birth"] == "Female" else 1.0) for i in IDS})
    steps.append(("legacy cohort (used here)", len(IDS)))

missing = [i for i in IDS if i not in cov]
assert not missing, f"{len(missing)} cohort IDs are not releasable PacBio in the covariates, e.g. {missing[:3]}"
absent = [i for i in IDS if i not in pheno.index]
assert not absent, f"{len(absent)} cohort IDs are not in the phenotype table, e.g. {absent[:3]}"
GROUP = pd.Series({i: cov[i]["group"] for i in IDS})

print(pd.DataFrame(steps, columns=["step", "n"]).to_string(index=False))

comp = pd.DataFrame({
    "participants": GROUP.value_counts().reindex(GROUPS).fillna(0).astype(int),
    "female": pd.Series({g: int(((GROUP == g) & (SEX == 0.0)).sum()) for g in GROUPS}),
    "male": pd.Series({g: int(((GROUP == g) & (SEX == 1.0)).sum()) for g in GROUPS}),
})
comp.loc["Total"] = comp.sum()
comp["female %"] = (100 * comp["female"] / comp["participants"]).round(1)
print(f"\nanalysis cohort composition ({COHORT}):")
print(comp.to_string())
comp.to_csv(TABLES / "cohort_composition.tsv", sep="\t")

## 3. Encode: code burden and per-code counts

`CASE` and `ELIG` (cohort × code) are reused by sections 8-11.

In [ ]:
n, m = len(IDS), len(PHE_COLS)
CASE = np.zeros((n, m), dtype=bool)
ELIG = np.zeros((n, m), dtype=bool)
CASE_ALL_SEX = np.zeros((n, m), dtype=bool)  # before the sex restriction, for the section 10 audit

sub = pheno.loc[IDS]
sex_arr = SEX.reindex(IDS).to_numpy(float)
for j, col in enumerate(PHE_COLS):
    enc = sp.encode_binary(sub[col])
    CASE_ALL_SEX[:, j] = (enc == 1.0).fillna(False).to_numpy(bool)
    side = PHECODE_SEX.get(CODE_OF[col], "Both")
    if side in ("Male", "Female"):
        wrong = sex_arr == (0.0 if side == "Male" else 1.0)  # sex is 0 = female, 1 = male
        enc = enc.mask(pd.Series(wrong, index=enc.index), pd.NA)
    CASE[:, j] = (enc == 1.0).fillna(False).to_numpy(bool)
    ELIG[:, j] = enc.notna().to_numpy(bool)

burden = CASE.sum(axis=1)
per_code = pd.DataFrame({
    "phecode": [CODE_OF[c] for c in PHE_COLS],
    "n_cases": CASE.sum(axis=0),
    "n_eligible": ELIG.sum(axis=0),
}).assign(prevalence=lambda d: d["n_cases"] / d["n_eligible"].where(d["n_eligible"] > 0))

print("codes per participant:")
print(pd.Series(burden).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(1).to_string())
print(f"participants with no code at all: {int((burden == 0).sum()):,}")
print("\ncases per phecode:")
print(per_code["n_cases"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1).to_string())
print(f"phecodes with >= {MIN_CASES} cases (pooled): {int((per_code['n_cases'] >= MIN_CASES).sum()):,}")
print(f"phecodes with 0 cases: {int((per_code['n_cases'] == 0).sum()):,}")
print("\neligible participants per phecode (sex-specific codes sit near half the cohort):")
print(per_code["n_eligible"].describe(percentiles=[0.05, 0.5, 0.95]).round(0).to_string())

fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.3))
axes[0].hist(burden, bins=np.arange(0, np.percentile(burden, 99.5) + 2), color="#2F4858", lw=0)
axes[0].axvline(float(np.median(burden)), color="#C4473D", lw=1.0, ls="--")
axes[0].set_xlabel(f"Phecodes per participant (median {np.median(burden):.0f})")
axes[0].set_ylabel("Participants")
pos = per_code.loc[per_code["n_cases"] > 0, "n_cases"]
axes[1].hist(np.log10(pos), bins=40, color="#2F4858", lw=0)
axes[1].axvline(np.log10(MIN_CASES), color="#C4473D", lw=1.0, ls="--")
axes[1].set_xlabel(f"log10 cases per phecode (dashed: {MIN_CASES})")
axes[1].set_ylabel("Phecodes")
for ax in axes:
    for s_ in ("top", "right"):
        ax.spines[s_].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES / "code_burden.png", dpi=300, bbox_inches="tight")
per_code.to_csv(TABLES / "per_code_counts.tsv", sep="\t", index=False)

## 4. Case counts — `make_fig3_case_counts.py`

In [ ]:
STEM = "fig3_phecode_case_counts_current" if COHORT == "selector" else "fig3_phecode_case_counts"
SUMMARY_STEM = "fig3_cohort_summary_current" if COHORT == "selector" else "fig3_cohort_summary"
COUNTS_PATH = FIG_DATA / f"{STEM}.tsv"
SUMMARY_PATH = FIG_DATA / f"{SUMMARY_STEM}.json"

sh(f"python3 {SCRIPTS / 'make_fig3_case_counts.py'}"
   f" --selection-dir {SELECTION_DIR}"
   f" --phenotype-csv {PHENOTYPE_CSV}"
   f" --covariates-csv {COVARIATES_CSV}"
   f" --phecode-sex-tsv {PHECODE_CSV}"
   f" --global-anc {GLOBAL_ANC}"
   f" --out-counts {COUNTS_PATH}"
   f" --out-summary {SUMMARY_PATH}"
   + (" --legacy-sex-at-birth" if COHORT == "legacy" else ""))

counts = pd.read_csv(COUNTS_PATH, sep="\t", dtype={"phecode": str})
summary = json.loads(SUMMARY_PATH.read_text())
assert summary["n"] == len(IDS), (summary["n"], len(IDS))
assert (counts[[f"cases_{g}" for g in GROUPS]].sum(axis=1) == counts["n_cases"]).all()

# Section 3 encoded the table independently; the two must agree, or the
# exploratory views and the figure inputs describe different datasets.
assert list(counts["phecode"]) == [CODE_OF[c] for c in PHE_COLS], \
    "the counts table is not in phenotype-column order; the comparison below would be meaningless"
d_cases = int(np.abs(counts["n_cases"].to_numpy() - per_code["n_cases"].to_numpy()).max())
d_elig = int(np.abs(counts["n_eligible"].to_numpy() - per_code["n_eligible"].to_numpy()).max())
assert d_cases == 0 and d_elig == 0, f"notebook vs script disagree: cases {d_cases}, eligible {d_elig}"
print(f"\n{len(counts):,} phecodes x {summary['n']:,} participants   [{COHORT}]")
print(f"notebook encoding agrees with {COUNTS_PATH.name} on every case and eligible count")
print(json.dumps(summary, indent=1))

## 5. Cases by chapter, and the best-powered codes

In [ ]:
cdf = counts.copy()
cdf["category"] = cdf["phecode"].map(CHAPTER)
cdf["name"] = cdf["phecode"].map(CODE_NAME)
cdf["sex_rule"] = cdf["phecode"].map(PHECODE_SEX)
no_chapter = cdf.loc[cdf["category"].isna(), "phecode"].tolist()
assert not no_chapter, f"{len(no_chapter)} codes have no PhecodeX chapter, e.g. {no_chapter[:5]}"

byc = cdf.groupby("category")
chap = pd.DataFrame({
    "codes_in_table": byc.size(),
    "codes_defined": CHAPTER_SIZE,
    f"testable_{MIN_CASES}": byc["n_cases"].apply(lambda s: int((s >= MIN_CASES).sum())),
    "median_cases": byc["n_cases"].median(),
    "max_cases": byc["n_cases"].max(),
}).fillna(0).sort_values(f"testable_{MIN_CASES}", ascending=False)
chap.loc["Total"] = chap.sum()
chap.loc["Total", "median_cases"] = float(cdf["n_cases"].median())
chap.loc["Total", "max_cases"] = int(cdf["n_cases"].max())
print(chap.astype({"codes_in_table": int, "codes_defined": int, f"testable_{MIN_CASES}": int,
                   "max_cases": int}).round(1).to_string())
chap.to_csv(TABLES / "chapter_summary.tsv", sep="\t")

top = cdf.sort_values("n_cases", ascending=False).head(25)[
    ["phecode", "name", "category", "n_cases", "n_eligible", "sex_rule"]]
top["prevalence_%"] = (100 * top["n_cases"] / top["n_eligible"]).round(1)
print("\n25 best-powered phecodes:")
print(top.to_string(index=False))
cdf.sort_values("n_cases", ascending=False).to_csv(TABLES / "phecode_counts_annotated.tsv", sep="\t", index=False)

## 6. Figure 3A — testable phecodes per chapter and participant group

In [ ]:
cov_fig = S.build_coverage("testable", counts_path=COUNTS_PATH, summary_path=SUMMARY_PATH)
assert not cov_fig.simulated, "build_coverage fell back to simulated counts — the counts table was not read"
assert cov_fig.mode == "group", cov_fig.mode
assert cov_fig.n_total == int(sum(cov_fig.totals.values())), (cov_fig.n_total, cov_fig.totals)
pooled_cells = [cov_fig.testable[ch]["ALL"] for ch in cov_fig.chapters]
assert all(a >= b for a, b in zip(pooled_cells, pooled_cells[1:])), "chapter columns must be sorted by the Joint cell"

matrix_h = 0.150 * (len(cov_fig.keys) + 2) + 0.04  # panel_matrix_h's own row geometry
fig = new_figure(M.A_TOP + matrix_h + 0.12)
th = panel_matrix_h(fig, cov_fig, left=1.30, top=M.A_TOP, style=I.count_style(cov_fig), pooled_label="Joint",
                    cw=0.294, x_pop=-1.22, pooled_gap=0.0,
                    count_labels=("Participants with\nPacBio data", "In analysis"))
assert abs(th - matrix_h) < 1e-9, (th, matrix_h)
M._head(fig, "A", f"Testable phecodes by ancestry group and PhecodeX chapter "
                  f"({MIN_CASES} cases, {COHORT} cohort, n = {cov_fig.n_total:,})", 0.02, M.A_HEAD_Y)
for ext in ("png", "pdf"):
    fig.savefig(FIGURES / f"fig3a_testable_matrix.{ext}", dpi=400, bbox_inches="tight", pad_inches=0.04)

A = pd.DataFrame({k: {ch: cov_fig.testable[ch][k] for ch in cov_fig.chapters} for k in GROUPS + ["ALL"]})
A = A.loc[list(cov_fig.chapters)]
A.insert(0, "codes_defined", pd.Series(cov_fig.n_in_chapter).reindex(A.index))
A.loc["Total"] = A.sum()
A = A.astype(int)
A.index.name = "chapter"
A.to_csv(TABLES / "fig3a_testable_matrix.tsv", sep="\t")
print(A.to_string())

## 7. Figure 3B — testable phecodes against the case cutoff

The exported table has the columns of `figS_phecode_cutoff.tsv`.

In [ ]:
counts_rows = S.load_case_counts(COUNTS_PATH)
fig = new_figure(2.15, 3.45)
ax = fig.add_axes([0.165, 0.165, 0.800, 0.760])
pooled_curve = I.panel_cutoff(ax, counts_rows)
for ext in ("png", "pdf"):
    fig.savefig(FIGURES / f"fig3b_cutoff_curve.{ext}", dpi=400, bbox_inches="tight", pad_inches=0.04)

THR = np.arange(1, 1001)


def curve(values) -> np.ndarray:
    return (np.asarray(values, float)[None, :] >= THR[:, None]).sum(axis=1)


pooled = curve(cdf["n_cases"].to_numpy())
assert np.array_equal(pooled[:len(pooled_curve)], np.asarray(pooled_curve)), \
    "the exported curve differs from the one panel B drew"
group_curves = {g: curve(cdf[f"cases_{g}"].to_numpy()) for g in GROUPS}
any_group = curve(cdf[[f"cases_{g}" for g in GROUPS]].to_numpy().max(axis=1))
hap_curves = {f"hap_{k}": (cdf[f"hap_{k}"].to_numpy(float)[None, :] >= 2 * THR[:, None]).sum(axis=1)
              for k in LOCAL}  # the haplotype floor is 2x the case cutoff, so the curves share one x axis

cutoff_tbl = pd.DataFrame({"cutoff": THR, "pooled": pooled, "any_group": any_group,
                           **{f"group_{g}": group_curves[g] for g in GROUPS}, **hap_curves})
cutoff_tbl.to_csv(TABLES / "phecode_cutoff_curve.tsv", sep="\t", index=False)

i = MIN_CASES - 1
print(cutoff_tbl.set_index("cutoff").loc[[10, 20, MIN_CASES, 100, 200, 500]].to_string())
print(f"\nat the {MIN_CASES}-case floor: {pooled[i]:,} phecodes pooled, {any_group[i]:,} in at least one group")

## 8. Chapter prevalence per ancestry group

In [ ]:
chapters = list(cov_fig.chapters)
col_chapter = np.array([CHAPTER.get(CODE_OF[c], "") for c in PHE_COLS])
grp = GROUP.reindex(IDS).to_numpy()
prev = pd.DataFrame(index=chapters, columns=GROUPS + ["All"], dtype=float)
ncase = pd.DataFrame(index=chapters, columns=GROUPS + ["All"], dtype=float)
for ch in chapters:
    cols = col_chapter == ch
    if not cols.any():
        continue
    any_code = CASE[:, cols].any(axis=1)
    for g in GROUPS + ["All"]:
        sel_g = np.ones(len(IDS), bool) if g == "All" else (grp == g)
        k = int(any_code[sel_g].sum())
        ncase.at[ch, g] = k
        prev.at[ch, g] = 100.0 * k / max(int(sel_g.sum()), 1) if k >= MIN_PREVALENCE_CASES else np.nan

out = prev.round(1)
out.insert(0, "codes", [int((col_chapter == ch).sum()) for ch in chapters])
print(f"percent of the group with at least one code in the chapter (blank: < {MIN_PREVALENCE_CASES} cases)")
print(out.to_string())
prev.to_csv(TABLES / "chapter_prevalence_by_group.tsv", sep="\t")
ncase.to_csv(TABLES / "chapter_cases_by_group.tsv", sep="\t")

P = prev[GROUPS + ["All"]].to_numpy(float)
from matplotlib.colors import PowerNorm
pnorm = PowerNorm(0.5, vmin=0, vmax=float(np.nanmax(P)))
fig, ax = plt.subplots(figsize=(4.6, 0.26 * len(chapters) + 0.9))
ax.imshow(np.ma.masked_invalid(P), cmap=M.TAN_GREEN, aspect="auto", norm=pnorm)
ax.set_xticks(range(len(GROUPS) + 1))
ax.set_xticklabels(GROUPS + ["All"], fontsize=7)
for t, g in zip(ax.get_xticklabels(), GROUPS + ["All"]):
    t.set_color(ANC_TEXT.get(g, "#111111"))
ax.set_yticks(range(len(chapters)))
ax.set_yticklabels(chapters, fontsize=7)
for a_ in range(P.shape[0]):
    for b_ in range(P.shape[1]):
        if np.isfinite(P[a_, b_]):
            fc = M.TAN_GREEN(pnorm(P[a_, b_]))
            ax.text(b_, a_, f"{P[a_, b_]:.0f}", ha="center", va="center", fontsize=6.5,
                    color="#FFFFFF" if M._lum(fc) < 0.30 else "#111111")
ax.set_xticks(np.arange(-0.5, P.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, P.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", lw=0.8)
ax.tick_params(which="minor", length=0)
ax.set_title("% of group with any code in the chapter", fontsize=7, color="#555555", pad=4)
fig.savefig(FIGURES / "chapter_prevalence_by_group.png", dpi=300, bbox_inches="tight")

## 9. Local-ancestry testability

Expected case haplotypes from chr1 global ancestry (2 × the sum of a case's
proportion), not counts of genome-wide local-ancestry calls.

In [ ]:
hap = pd.DataFrame({k: {ch: int((cdf.loc[cdf["category"] == ch, f"hap_{k}"] >= MIN_HAP).sum())
                        for ch in chapters} for k in LOCAL}).loc[chapters]
hap["pooled"] = [int((cdf.loc[cdf["category"] == ch, "n_cases"] >= MIN_CASES).sum()) for ch in chapters]
hap.loc["Total"] = hap.sum()
print(f"phecodes with at least {MIN_HAP} expected case haplotypes of the local ancestry:")
print(hap.to_string())
hap.to_csv(TABLES / "fig3a_testable_matrix_local_ancestry.tsv", sep="\t")

tot = pd.Series(summary["hap_total"], dtype=float)
print("\nexpected case haplotypes in the cohort (chr1 global ancestry):")
print(pd.DataFrame({"haplotypes": tot.round(1), "% of 2n": (100 * tot / tot.sum()).round(1)}).to_string())
print(f"2n = {2 * summary['n']:,}")

## 10. Sex-specific audit — PhecodeX `sex` vs the observed case sex ratio

In [ ]:
n_f = CASE_ALL_SEX[sex_arr == 0.0].sum(axis=0)
n_m = CASE_ALL_SEX[sex_arr == 1.0].sum(axis=0)
tot_cases = n_f + n_m
with np.errstate(invalid="ignore", divide="ignore"):
    frac_f = np.where(tot_cases > 0, n_f / np.maximum(tot_cases, 1), np.nan)

audit = pd.DataFrame({
    "phecode": [CODE_OF[c] for c in PHE_COLS],
    "name": [CODE_NAME.get(CODE_OF[c], "") for c in PHE_COLS],
    "sex_rule": [PHECODE_SEX.get(CODE_OF[c], "") for c in PHE_COLS],
    "cases_female": n_f, "cases_male": n_m, "frac_female": np.round(frac_f, 4),
})
audit["flag"] = ""
enough = audit["cases_female"] + audit["cases_male"] >= MIN_PREVALENCE_CASES
audit.loc[enough & (audit["sex_rule"] == "Both") & (audit["frac_female"] >= 0.99), "flag"] = "Both but female-only"
audit.loc[enough & (audit["sex_rule"] == "Both") & (audit["frac_female"] <= 0.01), "flag"] = "Both but male-only"
audit.loc[(audit["sex_rule"] == "Female") & (audit["cases_male"] > 0), "flag"] = "Female rule, male cases present"
audit.loc[(audit["sex_rule"] == "Male") & (audit["cases_female"] > 0), "flag"] = "Male rule, female cases present"

print(audit["sex_rule"].value_counts().to_string())
print(f"\nsex-restricted phecodes with at least {MIN_CASES} cases: "
      f"{int(((audit['sex_rule'] != 'Both') & (tot_cases >= MIN_CASES)).sum()):,}")
flagged = audit[audit["flag"] != ""].sort_values("cases_female", ascending=False)
print(f"\n{len(flagged):,} codes disagree with the definitions table")
if len(flagged):
    print(flagged["flag"].value_counts().to_string())
    print()
    print(flagged.head(15).to_string(index=False))
audit.to_csv(TABLES / "sex_specific_audit.tsv", sep="\t", index=False)

## 11. Redundancy among testable phecodes

Pearson r on the 0/1 case vectors, missing treated as control; the selector's
own prune is pairwise-complete `corr(min_periods=50)`.

In [ ]:
R2_CUTOFF = float(os.environ.get("PHENO_MAX_R2", "0.5"))
keep_cols = np.where(CASE.sum(axis=0) >= MIN_CASES)[0]
codes = [CODE_OF[PHE_COLS[j]] for j in keep_cols]
print(f"{len(keep_cols):,} phecodes with at least {MIN_CASES} cases enter the redundancy check")

testable_set = set(codes)
parents = {c: sp.phecode_parent(c) for c in codes}
with_parent = [c for c, p in parents.items() if p in testable_set]
leaves = [c for c in codes if not any(parents.get(o, "") == c for o in codes)]
print(f"  {len(with_parent):,} have a testable parent code; {len(leaves):,} are leaves (no testable child)")

X = CASE[:, keep_cols].astype(np.float32)
X -= X.mean(axis=0, keepdims=True)
sd = np.sqrt((X ** 2).sum(axis=0))
X /= np.where(sd > 0, sd, 1.0)
R2 = (X.T @ X).astype(np.float32) ** 2
np.fill_diagonal(R2, 0.0)

case_n = CASE[:, keep_cols].sum(axis=0)
order = np.argsort(-np.minimum(case_n, len(IDS) - case_n))  # the selector ranks by min(cases, controls)
kept_mask = np.zeros(len(keep_cols), bool)
dropped: list[tuple[str, str, float]] = []
for j in order:
    if kept_mask.any():
        idx = np.where(kept_mask)[0]
        b = int(idx[np.argmax(R2[j, idx])])
        if R2[j, b] >= R2_CUTOFF:
            dropped.append((codes[j], codes[b], float(R2[j, b])))
            continue
    kept_mask[j] = True

max_r2 = R2.max(axis=1)
print(f"  max r2 with another testable phecode: median {np.median(max_r2):.3f}, "
      f"90th pct {np.percentile(max_r2, 90):.3f}")
print(f"  pairs with r2 >= {R2_CUTOFF}: {int((np.triu(R2, 1) >= R2_CUTOFF).sum()):,}")
print(f"  greedy prune at r2 >= {R2_CUTOFF} would drop {len(dropped):,} of {len(keep_cols):,}, "
      f"leaving {int(kept_mask.sum()):,}")
drop_df = pd.DataFrame(dropped, columns=["dropped", "kept", "r2"]).sort_values("r2", ascending=False)
drop_df["dropped_name"] = drop_df["dropped"].map(CODE_NAME)
drop_df["kept_name"] = drop_df["kept"].map(CODE_NAME)
if len(drop_df):
    print("\nmost redundant pairs:")
    print(drop_df.head(12).to_string(index=False))
drop_df.to_csv(TABLES / "redundant_phecode_pairs.tsv", sep="\t", index=False)
pd.DataFrame({"phecode": codes, "max_r2": np.round(max_r2, 4),
              "parent": [parents[c] for c in codes],
              "parent_testable": [parents[c] in testable_set for c in codes]}).to_csv(
    TABLES / "phecode_redundancy.tsv", sep="\t", index=False)
del X, R2

## 12. Write-out and copy-back

In [ ]:
lines = [
    f"# Phenotype table summary ({COHORT} cohort)",
    "",
    f"* phenotype table: `{PHENOTYPE_URI}`",
    f"* covariates: `{COVARIATES_URI}`",
    f"* PhecodeX definitions: `{PHECODE_URI}`",
    f"* chr1 global ancestry: `{GLOBAL_ANC_URI}`",
    f"* cohort: {COHORT}, n = {summary['n']:,} "
    f"({summary['n_female']:,} female, {summary['n_male']:,} male)",
    f"* candidate phecodes in the table: {len(counts):,} of {len(CHAPTER):,} PhecodeX codes",
    f"* case floor: fig3_sim.MIN_CASES = {MIN_CASES} cases pooled or within a group; "
    f"{MIN_HAP} expected case haplotypes",
    "",
    "## Headline counts",
    "",
    f"* testable pooled: **{pooled[i]:,}** phecodes",
    f"* testable in at least one participant group: {any_group[i]:,}",
    "* testable within a group: " + ", ".join(f"{g} {group_curves[g][i]:,}" for g in GROUPS),
    "* testable per local ancestry (haplotype floor): "
    + ", ".join(f"{k} {int((cdf[f'hap_{k}'] >= MIN_HAP).sum()):,}" for k in LOCAL),
    f"* median phecodes per participant: {np.median(burden):.0f} "
    f"(5-95%: {np.percentile(burden, 5):.0f}-{np.percentile(burden, 95):.0f})",
    f"* greedy r2 >= {R2_CUTOFF} prune would leave {int(kept_mask.sum()):,} of the "
    f"{len(keep_cols):,} testable phecodes",
]
(WORK / "summary.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

print("\nfiles written:")
for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(WORK)}  ({p.stat().st_size / 1e3:,.1f} kB)")

if BUCKET_PREFIX:
    sh(f"gsutil -m rsync -r {WORK}/ {BUCKET_PREFIX}/")
    print("\nuploaded to", BUCKET_PREFIX)

src_dir = f"{BUCKET_PREFIX}/figure_data_root/data" if BUCKET_PREFIX else str(FIG_DATA)
print("\nFigure 3 inputs for the manuscript repo (aggregate counts, no participant-level data):")
for name in (COUNTS_PATH.name, SUMMARY_PATH.name):
    print(f"  {src_dir}/{name}  ->  aou-lr-phase-2-manuscript/data/{name}")
print("\nThen, in the code repo:  python3 -m manuscript_figures fig3_interim")
if COHORT != "selector":
    print("\nNOTE: this is the legacy covariate-derived cohort, not the cohort the FELIX scan is "
          "fitted on. Its tables belong to the first-version figure pair; do not mix them with "
          "the *_current files.")